In [1]:
# ============================================================
# ETAPA 3 — CÉLULA 1 v5: Garante download + mmap real
# ============================================================
!pip install medmnist -q
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from medmnist import PathMNIST
import os, time, gc, psutil, csv

np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
ARQ = "/root/.medmnist/pathmnist_224.npz"

# GARANTE o download: se o arquivo não existe, baixa via medmnist
if not os.path.exists(ARQ):
    print("Arquivo não encontrado, baixando via medmnist...")
    _ = PathMNIST(split="train", size=224, download=True)  # baixa o .npz
    print("Download concluído.")
else:
    print("Arquivo já existe em disco.")

# Abre com mmap real (fica no disco, não estoura RAM)
dados = np.load(ARQ, mmap_mode="r")
print("Chaves:", list(dados.keys()))
print(f"RAM após abrir (mmap): {psutil.virtual_memory().available/1e9:.1f} GB")

X_train_mm = dados["train_images"]; y_train_mm = dados["train_labels"]
X_val_mm   = dados["val_images"];   y_val_mm   = dados["val_labels"]
X_test_mm  = dados["test_images"];  y_test_mm  = dados["test_labels"]

class PathDatasetMM(Dataset):
    def __init__(self, X, y, indices=None):
        self.X = X; self.y = y
        self.indices = indices if indices is not None else range(len(X))
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)])
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        img = self.tf(np.array(self.X[j]))
        label = int(np.array(self.y[j]).flatten()[0])
        return img, label

TAM_SUBSET = 20000
rng = np.random.default_rng(42)
idx_sub = rng.choice(len(X_train_mm), size=TAM_SUBSET, replace=False).tolist()

train_ds = PathDatasetMM(X_train_mm, y_train_mm, idx_sub)
val_ds   = PathDatasetMM(X_val_mm, y_val_mm)
test_ds  = PathDatasetMM(X_test_mm, y_test_mm)

BATCH=64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
print(f"Treino: {len(train_ds)} | Val: {len(val_ds)} | Teste: {len(test_ds)}")
print(f"Batches: {len(train_loader)} / {len(val_loader)} / {len(test_loader)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

100%|██████████| 12.6G/12.6G [06:56<00:00, 30.3MB/s]


Download concluído.
Chaves: ['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']
RAM após abrir (mmap): 18.4 GB
Treino: 20000 | Val: 10004 | Teste: 7180
Batches: 313 / 157 / 113


In [2]:
# ============================================================
# ETAPA 3 — CÉLULA 2: Funções de treino e avaliação
# ============================================================
def avaliar(modelo, loader, device):
    modelo.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.long().to(device)
            preds = torch.argmax(modelo(imgs), dim=1)
            acertos += (preds == labels).sum().item()
            total += labels.size(0)
    modelo.train()
    return acertos / total

def treinar(modelo, train_loader, val_loader, device,
            epocas=5, lr=0.001, otimizador_nome="adamw", nome="modelo"):
    criterio = nn.CrossEntropyLoss()
    if otimizador_nome == "adamw":
        otimizador = optim.AdamW(modelo.parameters(), lr=lr)
    else:
        otimizador = optim.SGD(modelo.parameters(), lr=lr, momentum=0.9)
    hist = {"loss": [], "acc_val": []}
    print(f"\n--- Treinando {nome} | {epocas} épocas | {otimizador_nome} lr={lr} ---")
    for ep in range(epocas):
        t0 = time.time()
        modelo.train()
        perda_acum, n_batches = 0.0, 0
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.long().to(device)
            otimizador.zero_grad()
            perda = criterio(modelo(imgs), labels)
            perda.backward()
            otimizador.step()
            perda_acum += perda.item(); n_batches += 1
        loss_ep = perda_acum / n_batches
        acc_val = avaliar(modelo, val_loader, device)
        hist["loss"].append(loss_ep); hist["acc_val"].append(acc_val)
        print(f"  Época {ep+1}/{epocas} | perda {loss_ep:.4f} | acc val {acc_val:.4f} | {time.time()-t0:.0f}s")
    return hist

resultados = {}
print("Funções e dicionário de resultados prontos.")

Funções e dicionário de resultados prontos.


In [3]:
# ============================================================
# ETAPA 3 — CÉLULA 3: CNN própria (autoral)
# ============================================================
class CNNPropria(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.bloco1 = nn.Sequential(nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.bloco2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.bloco3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.pool_final = nn.AdaptiveAvgPool2d(1)
        self.classificador = nn.Sequential(nn.Flatten(), nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64,num_classes))
    def forward(self, x):
        x=self.bloco1(x); x=self.bloco2(x); x=self.bloco3(x); x=self.pool_final(x); return self.classificador(x)

torch.manual_seed(42)
cnn = CNNPropria(num_classes=9).to(device)
hist_cnn = treinar(cnn, train_loader, val_loader, device, epocas=5, lr=0.001, otimizador_nome="adamw", nome="CNN_propria")
resultados["CNN_propria"] = {"acc_val_final": hist_cnn["acc_val"][-1], "acc_val_melhor": max(hist_cnn["acc_val"]),
                             "params": sum(p.numel() for p in cnn.parameters())}
torch.save(cnn.state_dict(), "cnn_propria.pth")
print("CNN própria treinada e salva.")


--- Treinando CNN_propria | 5 épocas | adamw lr=0.001 ---
  Época 1/5 | perda 1.0030 | acc val 0.7126 | 71s
  Época 2/5 | perda 0.6465 | acc val 0.8458 | 71s
  Época 3/5 | perda 0.5467 | acc val 0.8537 | 71s
  Época 4/5 | perda 0.4645 | acc val 0.8562 | 71s
  Época 5/5 | perda 0.4181 | acc val 0.7216 | 71s
CNN própria treinada e salva.


In [4]:
# ============================================================
# ETAPA 3 — CÉLULA 4: Função para criar CNNs pré-treinadas
# ============================================================
def criar_modelo_pretreinado(nome, num_classes=9, congelar=True):
    if nome == "resnet50":
        modelo = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        if congelar:
            for p in modelo.parameters(): p.requires_grad = False
        modelo.fc = nn.Linear(modelo.fc.in_features, num_classes)
    elif nome == "efficientnet_b0":
        modelo = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        if congelar:
            for p in modelo.parameters(): p.requires_grad = False
        modelo.classifier[1] = nn.Linear(modelo.classifier[1].in_features, num_classes)
    elif nome == "mobilenet_v2":
        modelo = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        if congelar:
            for p in modelo.parameters(): p.requires_grad = False
        modelo.classifier[1] = nn.Linear(modelo.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Modelo desconhecido: {nome}")
    return modelo.to(device)

print("Função de modelos pré-treinados pronta.")

Função de modelos pré-treinados pronta.


In [5]:
# ============================================================
# ETAPA 3 — CÉLULA 5: 3 CNNs (Feature Extraction + Fine-tuning)
# ============================================================
cnns = ["resnet50", "efficientnet_b0", "mobilenet_v2"]
for nome in cnns:
    torch.manual_seed(42)
    modelo_fe = criar_modelo_pretreinado(nome, congelar=True)
    hist_fe = treinar(modelo_fe, train_loader, val_loader, device, epocas=5, lr=0.001,
                      otimizador_nome="adamw", nome=f"{nome}_FeatureExtraction")
    resultados[f"{nome}_FE"] = {"acc_val_final": hist_fe["acc_val"][-1], "acc_val_melhor": max(hist_fe["acc_val"])}
    torch.save(modelo_fe.state_dict(), f"{nome}_fe.pth")

    torch.manual_seed(42)
    modelo_ft = criar_modelo_pretreinado(nome, congelar=False)
    hist_ft = treinar(modelo_ft, train_loader, val_loader, device, epocas=5, lr=0.0001,
                      otimizador_nome="adamw", nome=f"{nome}_FineTuning")
    resultados[f"{nome}_FT"] = {"acc_val_final": hist_ft["acc_val"][-1], "acc_val_melhor": max(hist_ft["acc_val"])}
    torch.save(modelo_ft.state_dict(), f"{nome}_ft.pth")
print("\n3 CNNs treinadas (FE + FT).")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 163MB/s]



--- Treinando resnet50_FeatureExtraction | 5 épocas | adamw lr=0.001 ---
  Época 1/5 | perda 0.3630 | acc val 0.9469 | 120s
  Época 2/5 | perda 0.1809 | acc val 0.9556 | 119s
  Época 3/5 | perda 0.1531 | acc val 0.9589 | 120s
  Época 4/5 | perda 0.1376 | acc val 0.9614 | 120s
  Época 5/5 | perda 0.1329 | acc val 0.9642 | 119s

--- Treinando resnet50_FineTuning | 5 épocas | adamw lr=0.0001 ---
  Época 1/5 | perda 0.1592 | acc val 0.9786 | 283s
  Época 2/5 | perda 0.0432 | acc val 0.9864 | 284s
  Época 3/5 | perda 0.0276 | acc val 0.9884 | 284s
  Época 4/5 | perda 0.0244 | acc val 0.9911 | 283s
  Época 5/5 | perda 0.0205 | acc val 0.9889 | 284s
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 141MB/s]



--- Treinando efficientnet_b0_FeatureExtraction | 5 épocas | adamw lr=0.001 ---
  Época 1/5 | perda 0.4643 | acc val 0.9433 | 61s
  Época 2/5 | perda 0.2176 | acc val 0.9524 | 61s
  Época 3/5 | perda 0.1858 | acc val 0.9583 | 61s
  Época 4/5 | perda 0.1672 | acc val 0.9594 | 61s
  Época 5/5 | perda 0.1569 | acc val 0.9621 | 61s

--- Treinando efficientnet_b0_FineTuning | 5 épocas | adamw lr=0.0001 ---
  Época 1/5 | perda 0.3405 | acc val 0.9831 | 139s
  Época 2/5 | perda 0.0550 | acc val 0.9905 | 139s
  Época 3/5 | perda 0.0349 | acc val 0.9910 | 139s
  Época 4/5 | perda 0.0268 | acc val 0.9927 | 139s
  Época 5/5 | perda 0.0188 | acc val 0.9933 | 139s
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 149MB/s]



--- Treinando mobilenet_v2_FeatureExtraction | 5 épocas | adamw lr=0.001 ---
  Época 1/5 | perda 0.3963 | acc val 0.9493 | 52s
  Época 2/5 | perda 0.2057 | acc val 0.9573 | 52s
  Época 3/5 | perda 0.1765 | acc val 0.9612 | 52s
  Época 4/5 | perda 0.1589 | acc val 0.9627 | 52s
  Época 5/5 | perda 0.1574 | acc val 0.9613 | 52s

--- Treinando mobilenet_v2_FineTuning | 5 épocas | adamw lr=0.0001 ---
  Época 1/5 | perda 0.1962 | acc val 0.9822 | 112s
  Época 2/5 | perda 0.0512 | acc val 0.9884 | 112s
  Época 3/5 | perda 0.0278 | acc val 0.9909 | 111s
  Época 4/5 | perda 0.0179 | acc val 0.9865 | 112s
  Época 5/5 | perda 0.0156 | acc val 0.9890 | 112s

3 CNNs treinadas (FE + FT).


In [6]:
# ============================================================
# ETAPA 3 — CÉLULA 6: Vision Transformer (ViT-B/16)
# ============================================================
def criar_vit(num_classes=9, congelar=True):
    modelo = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    if congelar:
        for p in modelo.parameters(): p.requires_grad = False
    modelo.heads.head = nn.Linear(modelo.heads.head.in_features, num_classes)
    return modelo.to(device)

torch.manual_seed(42)
vit = criar_vit(num_classes=9, congelar=True)
hist_vit = treinar(vit, train_loader, val_loader, device, epocas=5, lr=0.001, otimizador_nome="adamw", nome="ViT_B16_FE")
resultados["ViT_B16_FE"] = {"acc_val_final": hist_vit["acc_val"][-1], "acc_val_melhor": max(hist_vit["acc_val"])}
torch.save(vit.state_dict(), "vit_b16_fe.pth")
print("ViT treinado e salvo.")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 153MB/s]



--- Treinando ViT_B16_FE | 5 épocas | adamw lr=0.001 ---
  Época 1/5 | perda 0.3186 | acc val 0.9531 | 443s
  Época 2/5 | perda 0.1411 | acc val 0.9596 | 443s
  Época 3/5 | perda 0.1137 | acc val 0.9668 | 443s
  Época 4/5 | perda 0.0981 | acc val 0.9698 | 444s
  Época 5/5 | perda 0.0873 | acc val 0.9710 | 444s
ViT treinado e salvo.


In [7]:
# ============================================================
# ETAPA 3 — CÉLULA 7: Grid 2 otimizadores × 3 LRs (ResNet50 FE)
# ============================================================
otimizadores = ["adamw", "sgd"]
learning_rates = [1e-2, 1e-3, 1e-4]
EPOCAS_GRID = 3
grid_resultados = {}
print("=== Busca de hiperparâmetros (ResNet50, Feature Extraction) ===")
for otim_nome in otimizadores:
    for lr in learning_rates:
        torch.manual_seed(42)
        modelo_grid = criar_modelo_pretreinado("resnet50", congelar=True)
        hist_grid = treinar(modelo_grid, train_loader, val_loader, device, epocas=EPOCAS_GRID,
                            lr=lr, otimizador_nome=otim_nome, nome=f"ResNet_{otim_nome}_lr{lr}")
        grid_resultados[f"{otim_nome}_lr{lr}"] = hist_grid["acc_val"][-1]
print("\n=== Resumo do grid (acurácia val final) ===")
for chave, acc in sorted(grid_resultados.items(), key=lambda x: -x[1]):
    print(f"  {chave:20s}: {acc:.4f}")
melhor = max(grid_resultados, key=grid_resultados.get)
print(f"\nMelhor combinação: {melhor} ({grid_resultados[melhor]:.4f})")

=== Busca de hiperparâmetros (ResNet50, Feature Extraction) ===

--- Treinando ResNet_adamw_lr0.01 | 3 épocas | adamw lr=0.01 ---
  Época 1/3 | perda 0.4198 | acc val 0.9363 | 120s
  Época 2/3 | perda 0.2693 | acc val 0.9316 | 120s
  Época 3/3 | perda 0.2962 | acc val 0.9403 | 119s

--- Treinando ResNet_adamw_lr0.001 | 3 épocas | adamw lr=0.001 ---
  Época 1/3 | perda 0.3630 | acc val 0.9469 | 120s
  Época 2/3 | perda 0.1809 | acc val 0.9556 | 120s
  Época 3/3 | perda 0.1531 | acc val 0.9589 | 120s

--- Treinando ResNet_adamw_lr0.0001 | 3 épocas | adamw lr=0.0001 ---
  Época 1/3 | perda 1.0835 | acc val 0.9112 | 121s
  Época 2/3 | perda 0.4558 | acc val 0.9305 | 121s
  Época 3/3 | perda 0.3336 | acc val 0.9360 | 121s

--- Treinando ResNet_sgd_lr0.01 | 3 épocas | sgd lr=0.01 ---
  Época 1/3 | perda 0.2984 | acc val 0.9478 | 122s
  Época 2/3 | perda 0.1641 | acc val 0.9571 | 122s
  Época 3/3 | perda 0.1443 | acc val 0.9604 | 122s

--- Treinando ResNet_sgd_lr0.001 | 3 épocas | sgd lr=0.00

In [8]:
# ============================================================
# ETAPA 3 — CÉLULA 8: Tabela comparativa + CSVs
# ============================================================
print("="*60)
print(f"{'Modelo':<28} {'Acc Val Final':>13} {'Acc Val Melhor':>15}")
print("="*60)
for nome, info in resultados.items():
    print(f"{nome:<28} {info.get('acc_val_final',0):>13.4f} {info.get('acc_val_melhor',0):>15.4f}")
print("="*60)

with open("results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["modelo","acc_val_final","acc_val_melhor"])
    for nome, info in resultados.items():
        w.writerow([nome, info.get("acc_val_final",0), info.get("acc_val_melhor",0)])

with open("results_grid.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["config","acc_val_final"])
    for chave, acc in grid_resultados.items():
        w.writerow([chave, acc])
print("\nResultados salvos em results.csv e results_grid.csv")

Modelo                       Acc Val Final  Acc Val Melhor
CNN_propria                         0.7216          0.8562
resnet50_FE                         0.9642          0.9642
resnet50_FT                         0.9889          0.9911
efficientnet_b0_FE                  0.9621          0.9621
efficientnet_b0_FT                  0.9933          0.9933
mobilenet_v2_FE                     0.9613          0.9627
mobilenet_v2_FT                     0.9890          0.9909
ViT_B16_FE                          0.9710          0.9710

Resultados salvos em results.csv e results_grid.csv
